# Chapter 19
## Bursting

### Brian2 compatibility

| Example | Brian2 status |
|---|---|
| `ELLIPSES` | Brian2 |
| `ERISIR_PLUS_SLOW_I_K` | Brian2 |
| `ERISIR_SHOW_SLOW_I_K` | n/a |
| `INAPIK_PLUS_SLOW_I_K` | Brian2 |
| `INAPIK_PLUS_SLOW_I_K_3D` | n/a |
| `INAPIK_PLUS_STRONG_SLOW_I_K` | Brian2 |
| `INAPIK_PLUS_WEAK_SLOW_I_K` | Brian2 |
| `INAPIK_SHOW_SLOW_I_K` | n/a |
| `SQUARE_WAVES` | Brian2 |

`*_SHOW_SLOW_I_K` and `*_3D` sub-examples are dynamical-systems analysis
(effective-current threshold overlays, a 3D limit-cycle phase portrait)
rather than spiking-neuron time traces, so they stay Python-only. The
remaining six sub-examples are all voltage traces of a single bursting
neuron and share just two underlying models: a reduced I_NaP+I_K neuron
(`INAPIK_*`, `SQUARE_WAVES`) and an Erisir-type neuron (`ERISIR_*`,
`ELLIPSES`), each with a slow K+ current added on top. `SQUARE_WAVES` and
`ELLIPSES` reuse the exact same trace as `INAPIK_PLUS_SLOW_I_K` and
`ERISIR_PLUS_SLOW_I_K` respectively, just annotated differently to
illustrate the bursting geometry.


In [ ]:
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np


def simulate_inapik_slow_k(g_k_slow, tau_n_slow=20 * b2.ms, i_ext=7.0 * b2.uA,
                            duration=100 * b2.ms):
    """Reduced I_NaP+I_K bursting neuron (instantaneous Na activation) with
    a slow K+ current added on top, in Brian2.

    Args:
        g_k_slow (Quantity): slow K+ conductance
        tau_n_slow (Quantity): slow K+ gate time constant
        i_ext (Quantity): constant external current
        duration (Quantity): simulation time

    Returns:
        StateMonitor: Brian2 StateMonitor with recorded field "v"
    """
    eqs = """
    m_inf = 1 / (1 + exp((-20*mV - v) / (15*mV))) : 1
    n_inf = 1 / (1 + exp((-25*mV - v) / (5*mV))) : 1
    n_slow_inf = 1 / (1 + exp((-20*mV - v) / (5*mV))) : 1

    membrane_Im = g_na*m_inf*(v_na-v) + g_k*n*(v_k-v)         + g_k_slow*n_slow*(v_k-v) + g_l*(v_l-v) + i_ext : amp

    dn/dt = (n_inf-n)/tau_n : 1
    dn_slow/dt = (n_slow_inf-n_slow)/tau_n_slow : 1
    dv/dt = membrane_Im/C : volt
    """

    neuron = b2.NeuronGroup(
        1, eqs, method="rk2", dt=0.01 * b2.ms,
        namespace={
            "C": 1.0 * b2.ufarad,
            "g_na": 20.0 * b2.msiemens,
            "g_k": 10.0 * b2.msiemens,
            "g_l": 8.0 * b2.msiemens,
            "v_na": 60.0 * b2.mV,
            "v_k": -90.0 * b2.mV,
            "v_l": -80.0 * b2.mV,
            "tau_n": 0.15 * b2.ms,
            "g_k_slow": g_k_slow,
            "tau_n_slow": tau_n_slow,
            "i_ext": i_ext,
        },
    )
    neuron.v = -70.0 * b2.mV
    neuron.n = 0.6
    neuron.n_slow = 0.0

    state_mon = b2.StateMonitor(neuron, "v", record=True)
    net = b2.Network(neuron, state_mon)
    net.run(duration)
    return state_mon


In [ ]:
def simulate_erisir_slow_k(g_k_slow, tau_n_slow=100 * b2.ms, i_ext=7.5 * b2.uA,
                           duration=1000 * b2.ms):
    """Erisir et al. fast-spiking neuron with a slow K+ current added on
    top, in Brian2.

    Args:
        g_k_slow (Quantity): slow K+ conductance
        tau_n_slow (Quantity): slow K+ gate time constant
        i_ext (Quantity): constant external current
        duration (Quantity): simulation time

    Returns:
        StateMonitor: Brian2 StateMonitor with recorded field "v"
    """
    eqs = """
    alphah = 0.0035 / exp(v / (24.186*mV)) / ms : Hz
    alpham = 40/mV * (75.5*mV - v) / (exp((75.5*mV - v) / (13.5*mV)) - 1.0) / ms : Hz
    alphan = (95*mV - v)/mV / (exp((95*mV - v) / (11.8*mV)) - 1.0) / ms : Hz

    betah = -0.017/mV * (v + 51.25*mV) / (exp(-(v + 51.25*mV) / (5.2*mV)) - 1.0) / ms : Hz
    betam = 1.2262 / exp(v / (42.248*mV)) / ms : Hz
    betan = 0.025 / exp(v / (22.222*mV)) / ms : Hz

    m = alpham / (alpham + betam) : 1
    n_slow_inf = 1 / (1 + exp((-20*mV - v) / (5*mV))) : 1

    membrane_Im = g_na*m**3*h*(v_na-v) + g_k*n**2*(v_k-v)         + g_k_slow*n_slow*(v_k-v) + g_l*(v_l-v) + i_ext : amp

    dh/dt = alphah*(1-h) - betah*h : 1
    dn/dt = alphan*(1-n) - betan*n : 1
    dn_slow/dt = (n_slow_inf-n_slow)/tau_n_slow : 1
    dv/dt = membrane_Im/C : volt
    """

    neuron = b2.NeuronGroup(
        1, eqs, method="rk2", dt=0.01 * b2.ms,
        namespace={
            "C": 1.0 * b2.ufarad,
            "g_na": 112.0 * b2.msiemens,
            "g_k": 224.0 * b2.msiemens,
            "g_l": 0.5 * b2.msiemens,
            "v_na": 60.0 * b2.mV,
            "v_k": -90.0 * b2.mV,
            "v_l": -70.0 * b2.mV,
            "g_k_slow": g_k_slow,
            "tau_n_slow": tau_n_slow,
            "i_ext": i_ext,
        },
    )
    neuron.v = -70.0 * b2.mV
    neuron.h = "alphah / (alphah + betah)"
    neuron.n = "alphan / (alphan + betan)"
    neuron.n_slow = "1 / (1 + exp((-20*mV - v) / (5*mV)))"

    state_mon = b2.StateMonitor(neuron, "v", record=True)
    net = b2.Network(neuron, state_mon)
    net.run(duration)
    return state_mon


### INAPIK_PLUS_SLOW_I_K

In [ ]:
sm = simulate_inapik_slow_k(g_k_slow=5.0 * b2.msiemens)
t, v = sm.t / b2.ms, sm.v[0] / b2.mV

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t, v, "-k", linewidth=2)
ax.set(xlabel="$t$ [ms]", ylabel="$v$ [mV]")
fig.tight_layout()


### INAPIK_PLUS_STRONG_SLOW_I_K

In [ ]:
sm = simulate_inapik_slow_k(g_k_slow=20.0 * b2.msiemens)
t, v = sm.t / b2.ms, sm.v[0] / b2.mV

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t, v, "-k", linewidth=2)
ax.set(xlabel="$t$ [ms]", ylabel="$v$ [mV]")
fig.tight_layout()


### INAPIK_PLUS_WEAK_SLOW_I_K

In [ ]:
sm = simulate_inapik_slow_k(g_k_slow=4.0 * b2.msiemens)
t, v = sm.t / b2.ms, sm.v[0] / b2.mV

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t, v, "-k", linewidth=2)
ax.set(xlabel="$t$ [ms]", ylabel="$v$ [mV]")
fig.tight_layout()


### SQUARE_WAVES

Same trace as `INAPIK_PLUS_SLOW_I_K`, with the slow (quasi-steady)
portions of the trajectory highlighted in red -- where the central
difference dv/dt is small.

In [ ]:
sm = simulate_inapik_slow_k(g_k_slow=5.0 * b2.msiemens)
t, v = sm.t / b2.ms, sm.v[0] / b2.mV
dt = float(t[1] - t[0])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t, v, "-k", linewidth=1)

v_left, v_right = v[:-2], v[2:]
ind = np.where(np.abs(v_right - v_left) / dt < 1)[0]
ax.plot(t[ind + 1], v[ind + 1], "-r", linewidth=2)

ax.set(xlabel="$t$ [ms]", ylabel="$v$ [mV]")
fig.tight_layout()


### ERISIR_PLUS_SLOW_I_K

In [ ]:
sm = simulate_erisir_slow_k(g_k_slow=1.5 * b2.msiemens)
t, v = sm.t / b2.ms, sm.v[0] / b2.mV

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, v, "-k", linewidth=2)
ax.set(xlim=(0, 1000), ylim=(-95, 55), xlabel="$t$ [ms]", ylabel="$v$ [mV]")
fig.tight_layout()


### ELLIPSES

Same trace as `ERISIR_PLUS_SLOW_I_K`, with the local voltage
maxima/minima envelope highlighted in red -- tracing out the
"ellipses" the sub-example is named after.

In [ ]:
sm = simulate_erisir_slow_k(g_k_slow=1.5 * b2.msiemens)
t, v = sm.t / b2.ms, sm.v[0] / b2.mV

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, v, "-k", linewidth=1)

v_left, v_center, v_right = v[:-2], v[1:-1], v[2:]
ind = np.where((v_center > v_left) & (v_center > v_right))[0]
ax.plot(t[ind + 1], v[ind + 1], "-r", linewidth=2)
ind = np.where((v_center < v_left) & (v_center < v_right))[0]
ax.plot(t[ind + 1], v[ind + 1], "-r", linewidth=2)

ax.set(xlim=(0, 1000), ylim=(-95, 55), xlabel="$t$ [ms]", ylabel="$v$ [mV]")
fig.tight_layout()
